In [1]:
import pandas as pd
import requests
import json

keywords_df = pd.read_csv("other/text/keywords.csv")
keywords_full = keywords_df['keyword'].tolist()

job_experience_threshold_years = 4.5
minimum_yearly_salary = 96000
resume_path = "other/text/resume.docx"

In [2]:
import pandas as pd
import glob
import os

# Define the path to your data folder
# Replace 'data_folder' with the actual path to your folder
folder_path = 'data' 

# Use glob to find all files ending with '.csv' in the specified folder
# The wildcard '*' matches any file name
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Create an empty list to store individual dataframes
df_list = []

# Loop through the list of file paths, read each one, and append to the list
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    df_list.append(df)

# Concatenate all dataframes in the list into a single dataframe
# ignore_index=True ensures a continuous index for the final combined data
combined_df = pd.concat(df_list, ignore_index=True)

# Optional: Display the first few rows of the combined dataframe

In [ ]:
from app import job_sourcing_pipeline
data = job_sourcing_pipeline(keywords_full, minimum_yearly_salary, job_experience_threshold_years, resume_path)

INFO:digistudio.crawlers.linkedin:Starting fetch for 138 items.


Successfully uploaded 28 new documents to 'jobs-full' (skipped 110 duplicates).


Fetching Jobs: 100%|██████████| 138/138 [00:42<00:00,  3.21it/s]
INFO:digistudio.crawlers.linkedin:Completed. Found 1333 jobs. Failed: 4
INFO:digistudio.crawlers.linkedin:Starting parallel scraping with 4 workers, 5 chunks
Processing chunks:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from app import categorize_jobs, process_jobs, job_sourcing_pipeline
from digistudio.processing.documents import docx_markdown
import ast

resume = docx_markdown(resume_path)
combined_df['metadata'] = combined_df['metadata'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
data = combined_df.to_dict(orient='records')

jobs = process_jobs(data, minimum_yearly_salary, job_experience_threshold_years, resume)

    --------------- ALGORITHM STATISTICS -------------
    (Filtered 293 out of 442 salary-matching jobs)

    Salary Match Rate: 1.81% 
    (Filtered 442 out of 24410 total jobs)

    Filtered Jobs: 1.20% 
    (Filtered 293 out of 24410 total jobs)
    --------------------------------------------------
    Total Jobs Processed: 24410 --- Total Jobs Matched: 293
    
--- 293 jobs passed filtering criteria ---
--- 293 jobs matched against resume ---


In [4]:
output = categorize_jobs(jobs)

--- Job Matching Summary ---
high: 19 jobs
med: 157 jobs
low: 117 jobs


In [5]:
from digistudio.processing.upload import upload_dict

#upload_dict(output, "jobs-display")

In [6]:
from digistudio.integrations.firebase import get_firebase_client
import pandas as pd

client = get_firebase_client()

import json

collection_name = "jobs-display"
data = output

if not isinstance(data, list):
    data = [data]


# Fetch existing payloads from collection to avoid duplicates
existing_payloads = set()
existing_docs = client.find(collection_name, {'stats': {'$exists': True}})
for doc in existing_docs:
    if 'stats' in doc:
        # Serialize stats dict to JSON string for hashable comparison
        existing_payloads.add(json.dumps(doc['stats'], sort_keys=True))

# Filter out records with duplicate stats or missing metadata
filtered_data = []
for record in data:
    if 'metadata' not in record:
        print(f"Warning: Record missing 'metadata' field and will be skipped: {record}")
        continue
    # Serialize record's stats for comparison
    record_payload_json = json.dumps(record['stats'], sort_keys=True)
    if record_payload_json not in existing_payloads:
        filtered_data.append(record)
    else:
        continue
        # print(f"Skipping duplicate stats: {record['stats']}")

# Perform bulk upload only for non-duplicate records
if filtered_data:
    ids = client.insert_many(collection_name, filtered_data)
    print(f"Successfully uploaded {len(ids)} new documents to '{collection_name}' (skipped {len(data) - len(filtered_data)} duplicates).")
else:
    print("All records were duplicates. Nothing to upload.")


All records were duplicates. Nothing to upload.


In [7]:
ids = client.insert_many(collection_name, data)